In [2]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook

In [2]:
source = r"C:\Divyesh\Technoval Forecasting\Sales - plates.xlsx"
sheet1 = "Detail1"
sheet2 = "Detail2"
output = r"C:\Divyesh\Technoval Forecasting\Merged_sales_data.parquet"

In [3]:
df1 = pd.read_excel(source, sheet_name=sheet1, header=2)
df2 = pd.read_excel(source, sheet_name=sheet2, header=2)


assert list(df1.columns) == list(df2.columns), "Headers differ"

In [4]:
combined_sales_data = pd.concat(
    [df1.assign(Source="Detail1"), df2.assign(Source="Detail2")],
    ignore_index=True,
)

# Rename columns in the dataframe before saving parquet
# (parquet has no Excel header row for openpyxl to edit)
rename_map = {
    "imprp0": "Product Family",
    "imprp3": "Thickness",
    "CustType": "Customer Type",
    "imprp1": "Manufacturer or Trading",
}
combined_sales_data = combined_sales_data.rename(columns=rename_map)

# Parquet needs one type per column. Excel exports often mix
# numbers and text (e.g. SubSystem: 0.30 and "Blank").
# Convert object columns to string so write succeeds.
object_cols = combined_sales_data.select_dtypes(include=["object"]).columns
combined_sales_data[object_cols] = combined_sales_data[object_cols].astype("string")

# much faster + far less RAM than to_excel
combined_sales_data.to_parquet(output, index=False)
print(f"Saved to {output}")
print(combined_sales_data.shape)
print("Columns:", combined_sales_data.columns.tolist())


C:\Users\Divyesh Parmar\AppData\Local\Temp\ipykernel_29592\1883696287.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = combined_sales_data.select_dtypes(include=["object"]).columns


Saved to C:\Divyesh\Technoval Forecasting\Merged_sales_data.parquet
(652942, 62)
Columns: ['Company', 'Channel', 'Territory', 'Customer Type', 'Pcode', 'Customer', 'GlClass', 'Product', 'Grup', 'System', 'SubSystem', 'sdlitm', 'Item', 'LFItemName', 'Packing', 'InvDate', 'Qty', 'NetAmt', 'GrossAmt', 'qtynos', 'Dollors', 'BaseAmt', 'ParentGroup', 'BrPlant', 'SRP3Code', 'SRP1Code', 'ShipToTerr', 'ShipToCust', 'SDSHAN', 'Qtr', 'imsrp1', 'imsrp2', 'imsrp3', 'imsrp4', 'imsrp5', 'imsrp6', 'imsrp7', 'imsrp8', 'imsrp9', 'imsrp0', 'Manufacturer or Trading', 'imprp2', 'Thickness', 'imprp4', 'imprp5', 'imprp6', 'imprp7', 'imprp8', 'imprp9', 'Product Family', 'size', 'abac01', 'abac02', 'abac03', 'abac04', 'abac08', 'abac10', 'abac11', 'PalletType', 'imeqty', 'PalletQty', 'Source']


In [7]:
# Keep Merged_sales_data.parquet unchanged.
# Write a new file that contains only the columns used for forecasting.

raw_file = r"C:\Divyesh\Technoval Forecasting\Merged_sales_data.parquet"
cleaned_file = r"C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.parquet"
cleaned_excel = r"C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.xlsx"

useful_columns = [
    "Company",
    "Territory",
    "Customer",
    "Customer Type",
    "InvDate",
    "NetAmt",
    "qtynos",
    "Product Family",
    "Manufacturer or Trading",
]

raw_data = pd.read_parquet(raw_file, columns=useful_columns)
print(f"Raw rows read: {len(raw_data)}")

cleaned_data = raw_data.copy()

text_columns = [
    "Company",
    "Territory",
    "Customer",
    "Customer Type",
    "Product Family",
    "Manufacturer or Trading",
]
for column in text_columns:
    cleaned_data[column] = (
        cleaned_data[column].astype("string").str.strip().replace("", pd.NA)
    )

# Keep the original invoice date. Year and Month are added for grouping and seasonality.
# Do not replace InvDate with the first day of the month; the day may still be useful to inspect.
cleaned_data["InvDate"] = pd.to_datetime(cleaned_data["InvDate"], errors="coerce")
cleaned_data["qtynos"] = pd.to_numeric(cleaned_data["qtynos"], errors="coerce")
cleaned_data["NetAmt"] = pd.to_numeric(cleaned_data["NetAmt"], errors="coerce")

before_drop = len(cleaned_data)
cleaned_data = cleaned_data.dropna(
    subset=["Company", "Customer", "Manufacturer or Trading", "InvDate", "qtynos", "NetAmt"]
)
print(
    f"Dropped {before_drop - len(cleaned_data)} rows with missing "
    "company, customer, MFG/TRD, date, quantity, or value"
)

# Year and month used later to aggregate each series to a monthly total.
# Created only after invalid dates are dropped, so every row has both values.
cleaned_data["Year"] = cleaned_data["InvDate"].dt.year.astype("int16")
cleaned_data["Month"] = cleaned_data["InvDate"].dt.month.astype("int8")

missing_family = int(cleaned_data["Product Family"].isna().sum())
missing_share = missing_family / len(cleaned_data)
print(f"Missing Product Family: {missing_family} rows ({missing_share:.2%})")
if missing_share < 0.05:
    cleaned_data = cleaned_data.dropna(subset=["Product Family"])
    print("Dropped rows with missing Product Family (under 5%).")
else:
    cleaned_data["Product Family"] = cleaned_data["Product Family"].fillna("Unknown")
    print("Labeled missing Product Family as Unknown.")

cleaned_data["Territory"] = cleaned_data["Territory"].fillna("Unknown")
cleaned_data["Customer Type"] = cleaned_data["Customer Type"].fillna("Unknown")

cleaned_data.to_parquet(cleaned_file, index=False)
cleaned_data.to_excel(cleaned_excel, sheet_name="Cleaned Sales", index=False)
print(f"Saved cleaned parquet: {cleaned_file}")
print(f"Saved cleaned excel: {cleaned_excel}")
print(f"Cleaned shape: {cleaned_data.shape}")
print("Columns:", cleaned_data.columns.tolist())
print("Raw file left unchanged:", raw_file)

Raw rows read: 652942
Dropped 514 rows with missing company, customer, MFG/TRD, date, quantity, or value
Missing Product Family: 3 rows (0.00%)
Dropped rows with missing Product Family (under 5%).
Saved cleaned parquet: C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.parquet
Saved cleaned excel: C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.xlsx
Cleaned shape: (652425, 11)
Columns: ['Company', 'Territory', 'Customer', 'Customer Type', 'InvDate', 'NetAmt', 'qtynos', 'Product Family', 'Manufacturer or Trading', 'Year', 'Month']
Raw file left unchanged: C:\Divyesh\Technoval Forecasting\Merged_sales_data.parquet


In [3]:
# Dates in the cleaned file look like "30-04-2018 00:00:00".
# The time is always midnight, so it is not useful.
# Day-month-year order is also easy to misread later (05-04-2018 could be 5 Apr or 4 May).
# This cell reads the files already saved above. It does not rebuild them from the raw extract.

cleaned_file = r"C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.parquet"
cleaned_excel = r"C:\Divyesh\Technoval Forecasting\Cleaned_sales_data.xlsx"

cleaned_data = pd.read_parquet(cleaned_file)

# dayfirst=True reads 30-04-2018 as 30 April 2018, not as an invalid month.
# errors="coerce" turns anything that is not a real date into missing.
invoice_dates = pd.to_datetime(cleaned_data["InvDate"], dayfirst=True, errors="coerce")

invalid_dates = int(invoice_dates.isna().sum())
if invalid_dates:
    cleaned_data = cleaned_data.loc[invoice_dates.notna()].copy()
    invoice_dates = invoice_dates.loc[invoice_dates.notna()]
    print(f"Dropped {invalid_dates} rows with an invalid invoice date")

# Store the date as text in year-month-day order, with no time.
# Example: 2018-04-30. This sorts correctly and has no DD/MM ambiguity.
# Year and Month are refreshed from the parsed date so they stay aligned with InvDate.
cleaned_data["Year"] = invoice_dates.dt.year.astype("int16")
cleaned_data["Month"] = invoice_dates.dt.month.astype("int8")
cleaned_data["InvDate"] = invoice_dates.dt.strftime("%Y-%m-%d")

cleaned_data.to_parquet(cleaned_file, index=False)
cleaned_data.to_excel(cleaned_excel, sheet_name="Cleaned Sales", index=False)

print(cleaned_data["InvDate"].head())
print(f"Rows: {len(cleaned_data)}")

C:\Users\Divyesh Parmar\AppData\Local\Temp\ipykernel_2860\1554098703.py:13: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  invoice_dates = pd.to_datetime(cleaned_data["InvDate"], dayfirst=True, errors="coerce")


0    2018-06-30
1    2018-04-30
2    2018-08-31
3    2018-10-31
4    2018-04-30
Name: InvDate, dtype: str
Rows: 652425
